In [5]:
"""
Frequency-Severity Reserving Engine.
Natively accepts and processes pandas DataFrames and Series.
"""

import logging
from typing import Dict, Any, List
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("PandasActuarialEngine")



class AdvancedFrequencySeverityModule:
    """
    Rigorously compliant Frequency-Severity reserving engine matching Friedland specifications.
    Accepts pandas DataFrames for triangles and Series for tracking vectors.
    """

    def __init__(
        self,
        reported_loss: pd.DataFrame,
        paid_loss: pd.DataFrame,
        reported_counts: pd.DataFrame,
        closed_counts: pd.DataFrame,
        case_outstanding: pd.Series,
        exposures: pd.Series,
        premium_rate_level_index: pd.Series
    ):
        """
        Initializes the workspace with pandas structures.
        The DataFrames are expected to have Accident Years as the Index
        and Development Maturities (Months) as the Column headers.
        """
        # Ensure identical index and column layouts across triangles
        self.ay = reported_loss.index.to_numpy(dtype=np.int32)
        self.maturities = reported_loss.columns.to_numpy(dtype=np.int32)
        self.num_ay = len(self.ay)
        self.num_mat = len(self.maturities)

        # Cast internal computing variables directly to NumPy float64 via pandas
        self.rep_loss = reported_loss.to_numpy(dtype=np.float64)
        self.paid_loss = paid_loss.to_numpy(dtype=np.float64)
        self.rep_counts = reported_counts.to_numpy(dtype=np.float64)
        self.closed_counts = closed_counts.to_numpy(dtype=np.float64)

        # Align 1D vectors with the specified Accident Year sorting index
        self.case_os = case_outstanding.loc[reported_loss.index].to_numpy(dtype=np.float64)
        self.exposures = exposures.loc[reported_loss.index].to_numpy(dtype=np.float64)
        self.rate_idx = premium_rate_level_index.loc[reported_loss.index].to_numpy(dtype=np.float64)

    def calculate_volume_weighted_factors(self, triangle: np.ndarray) -> np.ndarray:
        """Computes true, volume-weighted actuarial link ratios."""
        factors = np.ones(self.num_mat - 1)
        for j in range(self.num_mat - 1):
            denominators = []
            numerators = []
            for i in range(self.num_ay - 1 - j):
                if not np.isnan(triangle[i, j]) and not np.isnan(triangle[i, j+1]) and triangle[i, j] > 0:
                    denominators.append(triangle[i, j])
                    numerators.append(triangle[i, j+1])
            factors[j] = np.sum(numerators) / np.sum(denominators) if denominators else 1.0
        return factors

    def calculate_medial_average_factors(self, triangle: np.ndarray, n_years: int = 5) -> np.ndarray:
        """Implements textbook-specified medial averages (drops high/low extremes)."""
        factors = np.ones(self.num_mat - 1)
        for j in range(self.num_mat - 1):
            col_factors = []
            for i in range(self.num_ay - 1 - j):
                if triangle[i, j] > 0:
                    col_factors.append(triangle[i, j+1] / triangle[i, j])

            valid_factors = col_factors[-n_years:] if len(col_factors) >= n_years else col_factors
            if len(valid_factors) >= 3:
                valid_factors.sort()
                factors[j] = np.mean(valid_factors[1:-1])  # Drop lowest and highest extremes
            elif len(valid_factors) > 0:
                factors[j] = np.mean(valid_factors)
            else:
                factors[j] = 1.0
        return factors

    def build_cdf_vector(self, link_ratios: np.ndarray, tail_factor: float) -> np.ndarray:
        """Compounds link ratios backwards into cumulative development factors."""
        cdf = np.ones(self.num_mat)
        current_factor = tail_factor
        cdf[-1] = current_factor
        for j in range(self.num_mat - 2, -1, -1):
            current_factor *= link_ratios[j]
            cdf[j] = current_factor
        return cdf

    def check_monotonicity(self) -> List[bool]:
        """Validates that cumulative closed counts grow monotonically across maturities."""
        monotonicity_results = []
        for i in range(self.num_ay):
            row = self.closed_counts[i, :self.num_mat - i]
            if len(row) <= 1:
                monotonicity_results.append(True)
                continue
            monotonicity_results.append(bool(np.all(np.diff(row) >= 0)))
        return monotonicity_results

    def run_engine(self, config: Dict[str, Any]) -> pd.DataFrame:
        sev_trend = config.get('severity_trend', 0.05)
        freq_trend = config.get('frequency_trend', -0.015)
        tail_count = config.get('tail_factor_counts', 1.0)
        tail_sev = config.get('tail_factor_severity', 1.005)

        monotonic_flags = self.check_monotonicity()
        if not all(monotonic_flags):
            logger.warning("ALERT: Non-monotonic closed count development tracked in historical squares.")

        # --- Count Projections Core ---
        rep_count_links = self.calculate_volume_weighted_factors(self.rep_counts)
        closed_count_links = self.calculate_volume_weighted_factors(self.closed_counts)

        rep_counts_cdf = self.build_cdf_vector(rep_count_links, tail_count)
        closed_counts_cdf = self.build_cdf_vector(closed_count_links, tail_count)

        ult_counts_from_reported = np.zeros(self.num_ay)
        ult_counts_from_closed = np.zeros(self.num_ay)
        selected_ultimate_counts = np.zeros(self.num_ay)

        for i in range(self.num_ay):
            latest_j = self.num_mat - 1 - i
            ult_counts_from_reported[i] = self.rep_counts[i, latest_j] * rep_counts_cdf[latest_j]
            ult_counts_from_closed[i] = self.closed_counts[i, latest_j] * closed_counts_cdf[latest_j]

            if i < self.num_ay - 1:
                selected_ultimate_counts[i] = ult_counts_from_reported[i]
            else:
                selected_ultimate_counts[i] = (ult_counts_from_reported[i] + ult_counts_from_closed[i]) / 2

        latest_reported_loss = np.array([self.rep_loss[i, self.num_mat - 1 - i] for i in range(self.num_ay)])

        # --- APPROACH 1: REMEDIATED DEVELOPMENT ---
        hist_severities = np.zeros_like(self.rep_loss)
        valid_mask = self.rep_counts > 0
        hist_severities[valid_mask] = (self.rep_loss[valid_mask] * 1000.0) / self.rep_counts[valid_mask]

        sev_links = self.calculate_medial_average_factors(hist_severities, n_years=5)
        sev_cdf = self.build_cdf_vector(sev_links, tail_sev)

        ult_sev_app1 = np.zeros(self.num_ay)
        for i in range(self.num_ay):
            latest_j = self.num_mat - 1 - i
            ult_sev_app1[i] = hist_severities[i, latest_j] * sev_cdf[latest_j]

        ult_loss_app1 = (selected_ultimate_counts * ult_sev_app1) / 1000.0

        # --- APPROACH 2: EXPOSURE TRENDING ---
        anchor_idx = self.num_ay - 1
        on_level_premiums = self.exposures * self.rate_idx

        hist_frequencies = selected_ultimate_counts / on_level_premiums
        trended_frequencies = hist_frequencies * ((1.0 + freq_trend) ** (anchor_idx - np.arange(self.num_ay)))
        selected_base_freq = np.mean(trended_frequencies[:anchor_idx-1])

        trended_severities = ult_sev_app1 * ((1.0 + sev_trend) ** (anchor_idx - np.arange(self.num_ay)))
        selected_base_sev = np.mean(trended_severities[:anchor_idx-1])

        ult_loss_app2 = np.zeros(self.num_ay)
        for i in range(self.num_ay):
            if i >= self.num_ay - 2:
                projected_freq = selected_base_freq / ((1.0 + freq_trend) ** (anchor_idx - i))
                projected_sev = selected_base_sev / ((1.0 + sev_trend) ** (anchor_idx - i))
                ult_loss_app2[i] = (on_level_premiums[i] * projected_freq * projected_sev) / 1000.0
            else:
                ult_loss_app2[i] = ult_loss_app1[i]

        # --- APPROACH 3: INCREMENTAL DISPOSAL RATE ---
        disposal_matrix = np.zeros_like(self.closed_counts)
        for i in range(self.num_ay):
            if selected_ultimate_counts[i] > 0:
                disposal_matrix[i, :] = self.closed_counts[i, :] / selected_ultimate_counts[i]

        selected_dr = np.array([np.nanmean(disposal_matrix[:self.num_ay - j, j]) for j in range(self.num_mat)])

        incremental_loss_square = np.zeros_like(self.paid_loss)
        for i in range(self.num_ay):
            latest_j = self.num_mat - 1 - i
            for j in range(self.num_mat):
                if j <= latest_j:
                    incremental_loss_square[i, j] = self.paid_loss[i, j] - (self.paid_loss[i, j-1] if j > 0 else 0.0)
                else:
                    prev_dr = selected_dr[j-1] if j > 0 else 0.0
                    proj_incremental_closed = selected_ultimate_counts[i] * (selected_dr[j] - prev_dr)

                    years_from_anchor = anchor_idx - i
                    historical_base_sev = selected_base_sev / ((1.0 + sev_trend) ** years_from_anchor)
                    incremental_loss_square[i, j] = (proj_incremental_closed * historical_base_sev) / 1000.0

        ult_loss_app3 = np.sum(incremental_loss_square, axis=1)

        # Model routing strategy select toggle
        selected_approach = config.get('target_approach', 'classical_development')
        if selected_approach == 'exposure_trending':
            final_ult = ult_loss_app2
        elif selected_approach == 'incremental_disposal':
            final_ult = ult_loss_app3
        else:
            final_ult = ult_loss_app1

        # Calculate final padded vectors with a 0.0 safety floor
        ibnr = np.maximum(final_ult - latest_reported_loss, 0.0)
        total_unpaid = ibnr + self.case_os

        # Compile and return tabular metrics indexed neatly by Accident Year
        output_df = pd.DataFrame(index=self.ay)
        output_df['Ultimate_Loss_Estimate'] = final_ult
        output_df['Latest_Reported_Loss'] = latest_reported_loss
        output_df['Projected_IBNR'] = ibnr
        output_df['Case_Outstanding'] = self.case_os
        output_df['Total_Unpaid_Reserve'] = total_unpaid

        return output_df

# 1. Setup row indexes and column maturity timelines
accident_years = [2022, 2023, 2024, 2025]
maturities = [12, 24, 36, 48]

# 2. Build pandas DataFrames for your triangles (rows=AY, columns=Maturity)
df_reported_loss = pd.DataFrame(
    [[1500, 2420, 2720, 3020],
     [1150, 1840, 2070, np.nan],
     [1650, 2640, np.nan, np.nan],
     [1740, np.nan, np.nan, np.nan]], index=accident_years, columns=maturities
)

df_paid_loss = pd.DataFrame(
    [[600, 1220, 1520, 1820],
     [460, 920, 1150, np.nan],
     [660, 1320, np.nan, np.nan],
     [700, np.nan, np.nan, np.nan]], index=accident_years, columns=maturities
)

df_reported_counts = pd.DataFrame(
    [[100, 105, 105, 105],
     [90,  92,  92,  np.nan],
     [110, 115, np.nan, np.nan],
     [105, np.nan, np.nan, np.nan]], index=accident_years, columns=maturities
)

df_closed_counts = pd.DataFrame(
    [[40, 75, 95, 105],
     [35, 70, 90, np.nan],
     [42, 80, np.nan, np.nan],
     [38, np.nan, np.nan, np.nan]], index=accident_years, columns=maturities
)

# 3. Build pandas Series for your vectors
series_case_os = pd.Series([0.0, 420.0, 650.0, 540.0], index=accident_years)
series_exposure = pd.Series([10000.0, 10500.0, 11000.0, 11500.0], index=accident_years)
series_rate_idx = pd.Series([1.198, 1.142, 1.020, 1.000], index=accident_years)

# 4. Initialize the code engine natively
engine = AdvancedFrequencySeverityModule(
    reported_loss=df_reported_loss,
    paid_loss=df_paid_loss,
    reported_counts=df_reported_counts,
    closed_counts=df_closed_counts,
    case_outstanding=series_case_os,
    exposures=series_exposure,
    premium_rate_level_index=series_rate_idx
)

# 5. Run and fetch results as an organized structured pandas DataFrame
selections_config = {"target_approach": "exposure_trending", "severity_trend": 0.05}
final_reserves_table = engine.run_engine(selections_config)

print(final_reserves_table)


      Ultimate_Loss_Estimate  Latest_Reported_Loss  Projected_IBNR  \
2022             3035.100000                3020.0       15.100000   
2023             2309.800368                2070.0      239.800368   
2024             2623.340061                2640.0        0.000000   
2025             2780.898286                1740.0     1040.898286   

      Case_Outstanding  Total_Unpaid_Reserve  
2022               0.0             15.100000  
2023             420.0            659.800368  
2024             650.0            650.000000  
2025             540.0           1580.898286  


DataFrames